In [ ]:
import h5py
import pandas as pd
import numpy as np
import os
from matplotlib import pyplot as plt
import vbn_utils as vbn
from vbn_utils import formatFigure

%matplotlib inline

In [ ]:
from notebook_utils import get_opto_responses_for_units, plot_raster

In [ ]:
high_res = True
if high_res:
    plt.rcParams['figure.dpi'] = 150
    plt.rcParams['savefig.dpi'] = 300
    plt.rcParams['font.size'] = 12
    plt.rcParams['pdf.fonttype'] = 42

    plt.rcParams['figure.facecolor'] = 'white'
    plt.rcParams['axes.facecolor'] = 'white'
    plt.rcParams['savefig.facecolor'] = 'white'  # affects clipboard copy too
    plt.rcParams['savefig.transparent'] = False

## Data loading

In [ ]:
#Paths to all of the useful supplemental tables and tensors
opto_tensor_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/vbn_opto_tensor_unit_grouped.hdf5"
unit_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_units_with_responsiveness.csv"

In [ ]:
units = pd.read_csv(unit_table_file)

## Example session raster

In [ ]:
from allensdk.brain_observatory.behavior.behavior_project_cache.\
    behavior_neuropixels_project_cache \
    import VisualBehaviorNeuropixelsProjectCache


cache_dir = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/vbn_s3_cache"


cache = VisualBehaviorNeuropixelsProjectCache.from_s3_cache(
            cache_dir=cache_dir)
cache.load_manifest('visual-behavior-neuropixels_project_manifest_v0.4.0.json')

session_table = cache.get_ecephys_session_table()
sst_sessions = session_table[session_table['genotype'].str.contains('Sst')]
session_id = sst_sessions.index.values[0]
session = cache.get_ecephys_session(
           ecephys_session_id=session_id)

In [ ]:
session_sst_units = units[(units['ecephys_session_id'] == session_id)&(units['SST'])]

In [ ]:
plt.rcParams['font.size'] = 16
opto_table = session.optotagging_table
long_pulse = opto_table.loc[(opto_table['duration']==opto_table['duration'].max())&
                            (opto_table['level']==opto_table['level'].max())]['start_time'].values

short_pulse = opto_table.loc[(opto_table['duration']==opto_table['duration'].min())&
                            (opto_table['level']==opto_table['level'].max())]['start_time'].values

example_cells = [1179688501,]
for ssu in example_cells:
    spikes = session.spike_times[ssu]
    fig, ax = plt.subplots(1,2)
    fig.suptitle(str(ssu))
    plot_raster(ax[0], spikes, short_pulse, time_before=0.01, time_after = 0.02)
    plot_raster(ax[1], spikes, long_pulse, time_before=0.1, time_after = 1.1)
    formatFigure(fig, ax[0], xLabel='Time from laser onset (s)', yLabel='Trial')
    formatFigure(fig, ax[1],xLabel='Time from laser onset (s)', yLabel='Trial')
    plt.tight_layout()

## Waveform duration distribution

In [ ]:
vis_units = vbn.get_unit_ids(units, 'VISall')
vis_units = units[units['unit_id'].isin(vis_units)]

In [ ]:
fig, ax = plt.subplots()
ax.hist(vis_units['waveform_duration'], bins=np.arange(0, 1, 0.04), color='k', edgecolor='w')
formatFigure(fig, ax, xLabel='Spike width (ms)', yLabel='Unit count')
ax.axvline(0.4, color='gray', linestyle='--')

## Cell type classification criteria

| **Cell Type** | **Criteria** |
|--------------|-------------------------|
| **SST** | • Recorded in **Sst-IRES-Cre;Ai32** mouse <br>• Z-scored pulse-evoked firing rate is **greater than 2**<br>• First-spike latency during the pulse stimulus is **less than 8 ms**<br>• First-spike jitter during the pulse stimulus is **less than 2 ms**<br>• Responsive for **at least 30%** of cosine stimulus duration |
| **VIP** | • Recorded in **Vip-IRES-Cre;Ai32** mouse <br>• Z-scored pulse-evoked firing rate is **greater than 2**<br>• First-spike latency during the pulse stimulus is **less than 8 ms**<br>• First-spike jitter during the pulse stimulus is **less than 2 ms**<br>• Responsive for **at least 30%** of cosine stimulus duration |
| **RS (Regular Spiking)** | • Waveform duration is **greater than 0.4 ms**<br>• **Does not** meet SST criteria<br>• **Does not** meet VIP criteria |
| **FS (Fast Spiking)** | • Waveform duration is **less than 0.4 ms**<br>• **Does not** meet SST criteria<br>• **Does not** meet VIP criteria |